## Оценка результатов моделирования

В этой практической работе будем использовать датасет Тайваньского центра по переливанию крови. В нём находится следующая информация по донорам:

*   Recency (rec) — количество месяцев, прошедшее с момента последнего донорства;
*   Frequency (freq) — общее количество донорств по донору;
*   Monetary (mon) — количество сданной крови в кубических сантиметрах;
*   Time (time) — количество месяцев, прошедшее с первого донорства;
*   Target (target) — сдал ли донор кровь в марте.

Задача в следующем: предсказать, сдаст ли донор кровь в марте, исходя из его входных параметров.

## Обязательные задачи

In [1]:
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv('../data/transfusion_main.csv')
print(df.shape)
df.head()

(648, 5)


,rec,freq,mon,time,target
0,2,50,12500,98,1
1,0,13,3250,28,1
2,1,16,4000,35,1
3,2,20,5000,45,1
4,1,24,6000,77,0


**Задача 0. Модель решающего дерева**

1. Изучите предоставленные данные из файла transfusion_main.csv. 
2. Поделите данные на треин/тест. 
3. Обучите модель решающего дерева с random_state=42 на тренировочной выборке. 
4. Сделайте предикт на тестовой выборке. 
5. Посчитайте метрики качества на тренировочной и тестовой выборках. 

Сделайте предположение, переобучена ваша модель или нет. Аргументируйте свой ответ.

In [3]:
X = df.drop(['target'], axis=1)
y = df.target

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train, y_train)
pred_train = dt.predict(x_train)
pred_test = dt.predict(x_test)

print(f'Точность на train: {accuracy_score(pred_train, y_train):.2f}\n\
Точность на test: {accuracy_score(pred_test, y_test):.2f}')

Точность на train: 0.94
Точность на test: 0.74


Да, модель действительно переобучилась. Результаты точности на тренировчной и тестовых выборка рознятся на 20% - это говорит о переобучении

**Задача 1. Проверка качества с помощью кросс-валидации**

Проверьте предсказательную способность модели дерева решений с помощью кросс-валидации. Исходя из метрик качества, полученных на кросс-валидации, сделайте вывод о предсказательной способности данной модели.

In [4]:
# Ваш код здесь
cross_val_score(dt, X,y, cv=3)

array([0.64814815, 0.73148148, 0.70833333])

Модель переобучилась, кросс валидация это только подтверждает. Дисперсия огромная, на одних данных модель "угадывает", а на других - нет. Дерево зазубрило все возможные ветви и просто переобучилось, мы можем попробовать это исправить, "обрезав" ветви покрутив параметр max_deph в меньшую сторолну

**Задача 2. Проверка качества на отложенной выборке**

В результате работы над задачей предсказания сдачи крови вы решили предложить модель дерева решений для использования в промышленном решении. Заказчик хочет проверить качество решения офлайн-тестированием. Для этого он выдал вам выборку данных, которую вы раньше не видели. Вам нужно замерить качество модели на выборке заказчика.

1. Обучите модель деревья решения на всем датасете `df`. 
2. Загрузите отложенную выборку из файла `tranfusioin_oot.csv`. 
3. Проверьте качество модели на этой выборке и на выборке для обучения. 

Сделайте вывод, переобучена ли данная модель.

In [5]:
# Обучим модель на всем датасете и сделаем предсказание

dt.fit(X,y)
pred_y = dt.predict(X)

In [6]:
# Загрузим отложенную выборку из файла 

df_oot = pd.read_csv('../data/transfusion_oot.csv')

In [7]:
# Вытащим переменные из отложенной выборки 

X_oot = df_oot.drop(['target'], axis=1)
y_oot = df_oot.target

In [8]:
pred_y_oot = dt.predict(X_oot)

In [9]:
print(f'Точность на наших данных:{accuracy_score(pred_y,y):.2f}')
print(f'Точность на отложенных данных:{accuracy_score(pred_y_oot,y_oot):.2f}')

Точность на наших данных:0.94
Точность на отложенных данных:0.54


Модель переобучена, все тоже самое что и ранее, разница около 20%

## Дополнительные задачи

1. Попробуйте изменить параметры дерева решений таким образом, чтобы модель показывала более стабильные значения метрик на кросс валидации на `df`.

2. Затем проверьте, улучшилась ли предсказательная способности модели на отложенной выборке из датасета `transfusion_oot.csv`.

3. Выведите значимости для каждой фичи из обученного дерева.

**Подсказка:** модель дерева решений склонна к переобучению, так как она стремится строить дерево максимальной глубины. То есть максимально заточить свои ответы под данные, которые она видит в тренировочной выборке. Уменьшить переобучение такой модели можно, настроив параметры, которые отвечают за глубину дерева. Уменьшая глубину обучаемого дерева, вы предотвращаете переобучение. Явление, когда мы искусственно уменьшаем дерево, называется pruning (стрижка дерева).

В случае, если в датасете слишком много фичей, указав слишком маленькую глубину дерева, вы рискуете недообучить модель. 

In [10]:
X = df.drop(['target'], axis=1)
y = df.target
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

dtc = DecisionTreeClassifier(random_state=42)

In [11]:
# Сетка параметров, направленная на упрощение модели
param_grid = {
    'max_depth': [3, 5, 7, 10],            # Ограничиваем вертикальный рост
    'min_samples_leaf': [2, 5, 10, 20],     # Не даем создавать слишком мелкие листы
    'min_samples_split': [5, 10, 20],       # Узел делится только если в нем много данных
    'criterion': ['gini', 'entropy']        # Можно проверить разные функции ошибки
}

In [12]:
grid_search_rf = GridSearchCV(
    estimator=dtc,
    param_grid=param_grid,
    cv=5,                  # Кросс-валидация обязательна для борьбы с переобучением
    scoring='accuracy',    # Или 'f1', зависит от задачи
    n_jobs=-1
)

grid_search_rf.fit(x_train, y_train)

print(f"Лучшие параметры: {grid_search_rf.best_params_}")

Лучшие параметры: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 5}


In [13]:
best_model = grid_search_rf.best_estimator_
print(f"Train score: {best_model.score(x_train, y_train):.3f}")
print(f"Test score: {best_model.score(x_test, y_test):.3f}")


Train score: 0.826
Test score: 0.800


In [14]:
cross_val_score(best_model, X, y)

array([0.62307692, 0.7       , 0.79230769, 0.7751938 , 0.79844961])

Благодаря подбору гиперпараметров через GridSearchCV проблему переобучения удалось решить: метрики на обучающей и тестовой выборках стабилизировались. Однако кросс-валидация выявила высокую дисперсию результатов, что указывает на неоднородность (нерепрезентативность) распределения данных в датасете. В рамках данной задачи перемешивание выборки не проводилось, однако в реальных условиях это было бы необходимым шагом для повышения надежности модели. Текущую модель можно считать работоспособной, учитывая риски колебания точности на различных сегментах данных